# Loading and merging raw data and filtering for English-only

In [ ]:
%pip install langdetect
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install seaborn
%pip install nltk

In [20]:
import pandas as pd
import numpy as np
import glob
import os
import re
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.util import bigrams
from datetime import datetime

In [22]:
# Source data is available on X website: https://x.com/i/communitynotes/download-data

# === CONFIGURATION ===
# Replace these paths with the locations of your source data files

# Path to the main notes file
NOTES_DATA_PATH = os.path.expanduser('~/path/to/your/notes-00000.tsv')
notes = pd.read_csv(NOTES_DATA_PATH, sep='\t')

# Path to the note status history file
STATUS_HISTORY_PATH = os.path.expanduser('~/path/to/your/noteStatusHistory-00000.tsv')
status_history = pd.read_csv(STATUS_HISTORY_PATH, sep='\t')

# Path to the directory containing your ratings files (e.g., ratings-00000.tsv, ratings-00001.tsv, etc.)
RATINGS_DATA_DIR = os.path.expanduser('~/path/to/your/ratings/')


/var/folders/z0/qzq8g2qx44lcdrv4wy390xdc0000gs/T/ipykernel_4010/2549655303.py:2: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  notes = pd.read_csv('~/Desktop/Community Notes/original files from x/notes-00000.tsv', sep='\t') # data downloaded from X on 11/14/24
/var/folders/z0/qzq8g2qx44lcdrv4wy390xdc0000gs/T/ipykernel_4010/2549655303.py:3: DtypeWarning: Columns (10,19) have mixed types. Specify dtype option on import or set low_memory=False.
  status_history = pd.read_csv('~/Desktop/Community Notes/original files from x/noteStatusHistory-00000.tsv', sep='\t')


In [24]:
notes = notes.astype(str)
status_history = status_history.astype(str)

notesandstatus = notes.merge(status_history, on='noteId', how='outer')

columns_to_keep = [
    "noteId",
    "noteAuthorParticipantId_x",
    "createdAtMillis_x",
    "tweetId",
    "classification",
    "summary",
    "timestampMillisOfFirstNonNMRStatus",
    "firstNonNMRStatus"
    
]

notesandstatus = notesandstatus[columns_to_keep]

In [26]:
# *this cell takes about 45 minutes to run*

notesandstatus['has_summary'] = notesandstatus['summary'].notna() & notesandstatus['summary'].str.strip().ne("")
notesandstatus['summary'] = notesandstatus['summary'].astype(str)

from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 0

def is_english(text):
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False 

notesandstatus['is_english'] = notesandstatus['summary'].apply(is_english)

In [40]:
# Looping through each ratings file to pull together all of the note ratings for each note 
# Saves to working directory for later use in Ratings Count section of 01_data_analysis
# Takes about 10 minutes to run:

all_ratings = pd.DataFrame()

for file in glob.glob(os.path.join(RATINGS_DATA_DIR, "ratings-*.tsv")):
    df_file = pd.read_csv(file, sep='\t')
    
    all_ratings = pd.concat([all_ratings, df_file], ignore_index=True)

all_ratings.to_csv("all_ratings.csv", index=False)

# Pull out features

In [31]:
# Create new columns in df ('got_shown', 'hours_to_show', 'is_followedup').

df = notesandstatus

df['firstNonNMRStatus'] = df['firstNonNMRStatus'].replace('nan', np.nan)
df['is_followedup'] = df['firstNonNMRStatus'].notna()

df['got_shown'] = (df['firstNonNMRStatus'] == 'CURRENTLY_RATED_HELPFUL').astype(int)

df['time_difference'] = None

df['timestampMillisOfFirstNonNMRStatus'] = df['timestampMillisOfFirstNonNMRStatus'].replace('nan', np.nan)

df['timestampMillisOfFirstNonNMRStatus'] = df['timestampMillisOfFirstNonNMRStatus'].astype(float).astype('Int64')

df['createdAtMillis_x'] = df['createdAtMillis_x'].replace('nan', np.nan)

df['createdAtMillis_x'] = df['createdAtMillis_x'].astype(float).astype('Int64')

for index, row in df.iterrows():
    if row['got_shown'] == 1:
        df.at[index, 'time_difference'] = row['timestampMillisOfFirstNonNMRStatus'] - row['createdAtMillis_x']

df['hours_to_show'] = df['time_difference'] / (1000 * 60 * 60)

# Word and Bigram Count

In [ ]:
# What are the most common words in the "summary" column are for specific date range?

df_english = df[(df['is_english'] == True)

df_english.loc[:, 'created_date'] = pd.to_datetime(df_english['createdAtMillis_x'], unit='ms')

start_date = pd.Timestamp('2020-12-31')
end_date = pd.Timestamp('2024-11-06')  
   
df_filtered = df_english[(df_english['created_date'] >= start_date) & 
                         (df_english['created_date'] <= end_date)]

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

all_text = ' '.join(df_filtered['summary'].dropna().astype(str))

words = re.findall(r'\b[a-zA-Z]+\b', all_text.lower())

filtered_words = [word for word in words if word not in stop_words]

word_counts = Counter(filtered_words)

word_freq_df = pd.DataFrame({
    'word': list(word_counts.keys()),
    'frequency': list(word_counts.values())
})

word_freq_df_range = word_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

word_freq_df_range.to_csv('word_freq_df_range.csv', index=False)

In [ ]:
# counting bigrams (2 word phrases) in 'summary' corpus:

df_english.loc[:, 'created_date'] = pd.to_datetime(df_english['createdAtMillis_x'], unit='ms')

start_date = pd.Timestamp('2020-12-31')
end_date = pd.Timestamp('2024-11-06')

df_filtered = df_english[(df_english['created_date'] >= start_date) & 
                         (df_english['created_date'] <= end_date)]

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

all_text = ' '.join(df_filtered['summary'].dropna().astype(str))

words = re.findall(r'\b[a-zA-Z]+\b', all_text.lower())

filtered_words = [word for word in words if word not in stop_words]

bigrams_list = list(bigrams(filtered_words))

bigram_counts = Counter(bigrams_list)

bigram_freq_df = pd.DataFrame({
    'bigram': [' '.join(bigram) for bigram in bigram_counts.keys()],
    'frequency': list(bigram_counts.values())
})

bigram_freq_df_range = bigram_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

bigram_freq_df_range.to_csv('bigrams_count.csv', index=False)

In [ ]:
# Filtering for top keywords in is_political notes (I want to know what it's top constituentes are!)

df_english = df[df['is_english'] == True].copy()

df_english.loc[:, 'created_date'] = pd.to_datetime(df_english['createdAtMillis_x'], unit='ms')

start_date = pd.Timestamp('2020-12-31')
end_date = pd.Timestamp('2024-11-06')  
   
df_filtered = df_english[(df_english['created_date'] >= start_date) & 
                         (df_english['created_date'] <= end_date)]

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

df_political = df_filtered[df_filtered['is_political'] == True]

only_political_text = ' '.join(df_political['summary'].dropna().astype(str))

words = re.findall(r'\b[a-zA-Z]+\b', only_political_text.lower())

filtered_words = [word for word in words if word not in stop_words]

word_counts = Counter(filtered_words)

word_freq_df = pd.DataFrame({
    'word': list(word_counts.keys()),
    'frequency': list(word_counts.values())
})

word_freq_df = word_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

word_freq_df.to_csv('is_political_keywords.csv', index=False)

# Define keyword lists (topics), create new columns for each topic, and save

In [36]:
# Function takes a list of keywords (ie. "covid_keywords" and a keyword_name such as "covid"), 
# and creates a new column in df (ie. is_covid).

def add_keyword_column(df, keywords, keyword_name):
    def contains_keywords(text, keywords):
        text_lower = str(text).lower()
        return any(re.search(rf'\b{re.escape(keyword.lower())}\b', text_lower) for keyword in keywords)

    df[f'is_{keyword_name}'] = df['summary'].apply(lambda text: contains_keywords(text, keywords))
    tweet_ids = set(df[df[f'is_{keyword_name}']]['tweetId'])
    df[f'is_{keyword_name}'] |= df['tweetId'].isin(tweet_ids)

# keywords were collected in the following ways: my own ideas, gpt suggestions, gpt grouping of top words and bigrams from 'summary' corpus:
# many of these did not end up in final analyses - too much noise/unrelated content.

# 133 terms:
political_keywords = [
    "5G", "activist", "amendment", "anti-corruption", "anti-vax", "assembly", "autocracy",
    "authoritarian", "ballot", "biden", "campaign", "candidate", "capitalist", "caucus",
    "censorship", "city council", "civic", "civil liberties", "climate policy", "clinton",
    "congress", "congressman", "congresswoman", "conservative", "constitution", "conspiracy",
    "corruption", "council", "covid", "court", "debate", "democracy", "democrat", "disinformation",
    "diplomacy", "election", "elon musk", "equality", "european union", "executive", "federal",
    "foreign", "freedom", "gaza", "government", "governor", "hamas", "harris", "healthcare",
    "hezbollah", "hoax", "house of representatives", "immigration", "infrastructure", "iran",
    "israel", "israeli", "joe biden", "judiciary", "justice", "kamala harris", "kim jong un",
    "law", "left-wing", "legislation", "liberal", "lobbyist", "mandate", "mayor", "mcconnell",
    "middle east", "military", "misinformation", "movement", "mp", "musk", "nation",
    "national", "national security", "nationalism", "nato", "netanyahu", "news politics",
    "news world", "obama", "pac", "palestinian", "parliament", "party", "patriot", "patriotism",
    "pelosi", "plandemic", "policies", "policy", "political", "politics", "polling", "poverty",
    "power", "president", "progressive", "public office", "putin", "rally", "reform",
    "regulation", "relations", "representative", "republican", "rights", "russia", "sanctions",
    "schumer", "senate", "senator", "socialism", "socialist", "sovereignty", "starmer", "state",
    "states", "stimulus", "supreme court", "surveillance", "tariffs", "taxes", "transparency",
    "trump", "ukraine", "un", "united states", "vance", "veto", "vote", "walz", "waltz",
    "war", "welfare", "white house", "xi jinping", "zelensky"
]

#I left out "musk" because he didn't become a highly political figure until very near the end of the date range of this data.
political75_keywords = [
    "5G", "activist", "amendment", "anti-vax", "assembly", "authoritarian", "ballot", "biden",
    "campaign", "candidate", "caucus", "censorship", "city council", "civil liberties", 
    "climate policy", "congress", "conservative", "constitution", "conspiracy", "corruption",
    "covid", "covid19", "court", "debate", "democracy", "democrat", "disinformation", 
    "diplomacy", "election", "equality", "executive", "federal", "foreign", "freedom", 
    "gaza", "government", "governor", "hamas", "harris", "healthcare", "hoax", "immigration", 
    "infrastructure", "iran", "israel", "judiciary", "justice", "law", "left-wing", 
    "legislation", "liberal", "maga", "mandate", "mayor", "middle east", "military", 
    "misinformation", "movement", "national", "national security", "nato", "netanyahu", 
    "obama", "palestinian", "parliament", "party", "patriot", "policy", "politics", 
    "president", "reform", "republican", "rights", "russia", "trump"
]

#left out 'elon musk' - same reason as above.
political253_keywords = [
    "5G", "absentee ballot", "activist", "afghanistan", "ambassador", "amendment",
    "anti-corruption", "anti-vax", "anti-zionist", "antifa", "antisemitic", "antisemitism",
    "apartheid", "assembly", "asylum", "attorney general", "authoritarian", "autocracy",
    "ballot", "ballot box", "biden", "biological female", "biological male", "black lives matter",
    "border control", "bureaucracy", "cabinet", "campaign", "campaign finance", "candidate",
    "capitalist", "caucus", "censorship", "cia", "cis", "cisgender", "city council", "civic",
    "civil disobedience", "civil liberties", "civil rights", "climate policy", "clinton",
    "congress", "congressman", "congresswoman", "conservative", "conspiracy", "constitution",
    "constitutional", "corruption", "council", "court", "covid", "debate", "democracy", "democrat",
    "detransition", "diplomacy", "disinformation", "dnc", "election", "electoral college",
 "equality", "european union", "executive", "executive order", "fake news", "fbi",
    "federal", "foreign", "free palestine", "freedom", "from the river to the sea", "ftm", "g20",
    "g7", "gaza", "gaza strip", "gender", "gender dysphoria", "gender identity", "gender ideology",
    "gender-affirming", "geopolitics", "gerrymandering", "gop", "governance", "government",
    "government agency", "governor", "hamas", "harris", "healthcare", "hezbollah", "hoax",
    "house of representatives", "hunter biden", "idf", "immigration", "infrastructure",
    "international relations", "intifada", "iran", "iraq", "israel", "israel defense forces",
    "israel lobby", "israeli", "j6", "jan 6", "joe biden", "judiciary", "justice",
    "kamala harris", "kim jong un", "law", "lebanon", "left-wing", "legal reform", "legislation",
    "legislative", "liberal", "lobbyist", "maga", "mainstream media", "mandate", "march", "mayor",
    "mcconnell", "media bias", "middle east", "military", "misinformation", "movement", "mp",
    "mtf", "musk", "nation", "national", "national security", "nationalism", "nato", "netanyahu",
    "news politics", "news world", "non-binary", "nonbinary", "north korea", "obama", "occupation",
    "opec", "opinion piece", "oversight", "pac", "pakistan", "palestinian", "parliament", "party",
    "patriot", "patriotism", "pelosi", "plandemic", "policies", "policy", "political", "politics",
    "poll watchers", "polling", "poverty", "power", "president", "press freedom", "progressive",
    "pronouns", "propaganda", "prosecution", "protest", "puberty blockers", "public discourse",
    "public funds", "public office", "public policy", "putin", "rally", "red tape", "reform",
    "refugees", "regulation", "relations", "representative", "republican", "resistance", "rfk jr",
    "rights", "rule of law", "russia", "sanctions", "schumer", "scotus", "senate", "senator",
    "settlements", "sit-in", "social justice", "socialism", "socialist", "south korea",
    "sovereignty", "speech censorship", "starmer", "state", "states", "stimulus", "supreme court",
    "surveillance", "syria", "taiwan", "tariffs", "taxes", "terrorism", "trans", "trans man",
    "trans rights", "trans woman", "transgender", "transitioning", "transparency", "transphobia",
    "transphobic", "treaty", "trump", "two-state", "ukraine", "un", "united states", "vance",
    "veto", "vote", "voter ID", "voter fraud", "voter suppression", "waltz", "walz", "war",
    "welfare", "west bank", "white house", "xi jinping", "yemen", "zelensky", "zionism", "zionist"
]


# 74 bigrams in scam
scam_keywords = [
    "account impersonating", "account takeover", "aware risk", "Bitcoin scam", "blockchain fraud", "Cash App scam",
    "charity scam", "check fraud", "con", "counterfeit unfit", "credit card scam", 
    "crypto scam", "cryptocurrency scam", "debit card fraud", "deceptive", "deceptive site", "detector antivirus",
    "Ethereum scam", "ethereum phishing", "fake account", "fake fundraiser", 
    "fake job", "fake versions", "financial scam", "fishing", "fool people", "fraud", 
    "fraudulent", "get rich quick", "grift", "hoax", "identity theft", "impersonation", 
    "impersonating", "inheritance scam", "investment scam", "io scam", "IRS scam", 
    "job scam", "legal safety", "loan scam", "lottery scam", "malicious transactions", 
    "may counterfeit", "money laundering", "NFT scam", "payPal scam", "personal information",
    "phishing", "Ponzi", "potential threats", "pyramid scheme", "real account", 
    "recovery phrase", "risk products", "rug pull", "scam", "scams", 
    "scheme", "secret recovery", "spoofing", "stimulus check fraud", "stolen assets", 
    "swindle", "tax fraud", "theft malicious", "trading fraud", "transactions resulting", 
    "unfit use", "Venmo scam", "verified scam", "wire fraud", "work from home scam", 
    "Zelle scam"
]

crypto_keywords = ["crypto", "cryptocurrency", "ethereum", "metamask", "blockchain", 
                   "bitcoin", "digital currency", "web3", "decentralized", "nft", "token", 
                   "wallet", "crypto wallet", "crypto exchange", "smart contracts"]

crypto75_keywords = [
    "crypto", "cryptocurrency", "bitcoin", "ethereum", "blockchain", "metamask", 
    "digital currency", "web3", "decentralized", "nft", "token", "wallet", 
    "crypto wallet", "crypto exchange", "smart contracts", "altcoin", "defi", 
    "stablecoin", "binance", "coinbase", "kraken", "ftx", "ledger", "cold wallet", 
    "hot wallet", "hardware wallet", "seed phrase", "gas fees", "hashrate", "mining", 
    "staking", "airdrops", "yield farming", "liquidity pool", "private key", 
    "public key", "dapp", "rug pull", "pump and dump", "scam coin", "shitcoin", 
    "hodl", "moon", "to the moon", "bear market", "bull market", "tokenomics", 
    "layer 1", "layer 2", "on-chain", "off-chain", "solana", "cardano", "polkadot", 
    "chainlink", "dogecoin", "shiba inu", "litecoin", "bitcoin cash", "crypto trading", 
    "exchange", "DEX", "CEX", "ICO", "IDO", "KYC", "crypto tax", "block explorer", 
    "Etherscan", "Satoshi", "Satoshi Nakamoto", "block reward", "halving", "proof of work", 
    "proof of stake", "validator", "node", "governance token"
]

#this is exactly the same as above...just using a new name to make analysis clear
crypto75_notpolitical_keywords = [
    "crypto", "cryptocurrency", "bitcoin", "ethereum", "blockchain", "metamask", 
    "digital currency", "web3", "decentralized", "nft", "token", "wallet", 
    "crypto wallet", "crypto exchange", "smart contracts", "altcoin", "defi", 
    "stablecoin", "binance", "coinbase", "kraken", "ftx", "ledger", "cold wallet", 
    "hot wallet", "hardware wallet", "seed phrase", "gas fees", "hashrate", "mining", 
    "staking", "airdrops", "yield farming", "liquidity pool", "private key", 
    "public key", "dapp", "rug pull", "pump and dump", "scam coin", "shitcoin", 
    "hodl", "moon", "to the moon", "bear market", "bull market", "tokenomics", 
    "layer 1", "layer 2", "on-chain", "off-chain", "solana", "cardano", "polkadot", 
    "chainlink", "dogecoin", "shiba inu", "litecoin", "bitcoin cash", "crypto trading", 
    "exchange", "DEX", "CEX", "ICO", "IDO", "KYC", "crypto tax", "block explorer", 
    "Etherscan", "Satoshi", "Satoshi Nakamoto", "block reward", "halving", "proof of work", 
    "proof of stake", "validator", "node", "governance token"
]

health_keywords = [
    "ADHD", "AIDS", "CDC", "FDA", "HIV", "ICU", "NIH", "PTSD", "WHO", "acute", "adverse events",
    "allergy", "anxiety", "antibiotic", "antiviral", "asthma", "autism", "autoimmune", "bioweapon",
    "booster", "cancer", "cdc", "chronic", "clinic", "clinical", "clinical trial", "cold",
    "condition", "coronavirus", "covid", "covid vaccine", "covid vaccines", "data", "death",
    "depression", "diagnosis", "disease", "doctor", "double-blind", "drug", "efficacy", 
    "emergency room", "epidemic", "experiment", "fake cure", "flu", "health", "healthcare", 
    "heart disease", "hospital", "hydroxychloroquine", "illness", "influenza", "insulin", 
    "jab", "long covid", "medical", "medication", "medicine", "mental health", "microchip",
    "mRNA", "ncbi", "nih", "nlm", "nurse", "obesity", "outbreak", "pandemic", "patient", 
    "peer-reviewed", "placebo", "pmc", "public health", "research", "safety compliance", 
    "scientific", "side effects", "stroke", "study", "symptom", "treatment", "vaccine", 
    "vaccinated", "vaccination", "vaccines", "wellness"
]

science_keywords = [
    "science", "scientific", "scientist", "research", "study", "studies", "data",
    "evidence", "experiment", "experiments", "theory", "hypothesis", "peer-reviewed",
    "replication", "analysis", "results", "findings", "conclusion", "method", "methodology",
    "biology", "chemistry", "physics", "astronomy", "geology", "ecology", "genetics",
    "evolution", "neuroscience", "psychology", "sociology", "anthropology", "archaeology",
    "mathematics", "statistics", "climate science", "meteorology", "environmental science",
    "computer science", "engineering", "robotics", "artificial intelligence",
    "controlled trial", "clinical trial", "double-blind", "placebo", "observational",
    "correlation", "causation", "sample size", "significant", "statistically significant",
    "margin of error", "confidence interval", "bias", "variance", "model", "simulation",
    "journal", "publication", "preprint", "open access", "dataset", "citation",
    "peer review", "lab", "laboratory", "fieldwork", "NASA", "CDC", "NIH", "FDA",
    "NSF", "academic", "university", "institute",
    "evolution", "big bang", "quantum", "theory of relativity", "GMOs", "gene editing",
    "CRISPR", "AI", "machine learning", "falsifiable", "consensus", "pseudoscience"
]

israel_keywords = [
    "Hamas", "Al-Qassam Brigades", "Nukhba Forces", "Palestinian Islamic Jihad", "Al-Quds Brigades",
    "Hezbollah", "Kataib Hezbollah", "Asaib Ahl al-Haq", "Harakat al-Nujaba", "Liwa Fatemiyoun",
    "Liwa Zainabiyoun", "IRGC", "houthis", "Iron Dome", "Dahiya Doctrine", "Kill Zone",
    "Morag Corridor", "israel", "Netanyahu", "Nakba", "Intifada", "Right of Return",
    "Settlements", "Two-State Solution", "Palestinian Authority", "Fatah", "Gaza Blockade",
    "Occupied Territories"
]

israel75_keywords = [
    "Hamas", "Al-Qassam Brigades", "Nukhba Forces", "Palestinian Islamic Jihad", "Al-Quds Brigades",
    "Hezbollah", "Kataib Hezbollah", "Asaib Ahl al-Haq", "Harakat al-Nujaba", "Liwa Fatemiyoun",
    "Liwa Zainabiyoun", "IRGC", "houthis", "Iron Dome", "Dahiya Doctrine", "Kill Zone",
    "Morag Corridor", "israel", "Netanyahu", "Nakba", "Intifada", "Right of Return",
    "Settlements", "Two-State Solution", "Palestinian Authority", "Fatah", "Gaza Blockade",
    "Occupied Territories", "Gaza", "West Bank", "East Jerusalem", "Zionism", "Zionist",
    "Anti-Zionism", "Israeli Defense Forces", "IDF", "Shin Bet", "Mossad", "Israel-Hamas War",
    "October 7", "Hostages", "Ceasefire", "Truce", "Airstrike", "Bombardment", "Invasion",
    "Ground Operation", "Tunnel Network", "Martyrs", "Al-Aqsa Mosque", "Jerusalem", "Tel Aviv",
    "Sderot", "Ashkelon", "Rocket Attack", "Missile Defense", "Civilian Casualties",
    "Collective Punishment", "War Crimes", "Genocide", "Ethnic Cleansing", "Displacement",
    "Humanitarian Aid", "Refugees", "Gazan", "Palestinian", "Israeli", "Resistance",
    "Occupation", "Apartheid", "UNRWA", "UN Resolution", "Peace Process", "Oslo Accords",
    "BDS", "Boycott Divestment Sanctions", "Hague", "International Criminal Court"
]


trump_keywords = [
    "Trump", "Donald Trump", "realDonaldTrump", "Trump 2024", "Trump Train",
    "MAGA", "Make America Great Again", "Keep America Great", "Trump Rally",
    "Trump Supporters", "Trump Won", "Stop The Steal", "45th President", "47th President",
    "Trump Campaign", "Trump News", "Trump Speech", "Trump Admin", "Trump Administration",
    "Trump Indictment", "Trump Arrest", "Trump Trial", "Trump Judge",
    "Trump Family", "Ivanka Trump", "Melania Trump", "Eric Trump", "Don Jr",
    "Trump Organization", "Trump Hotel", "Trump Tower", "Trump Voters",
    "Trump Base", "Trumpism", "ProTrump", "AntiTrump", "mar-a-lago"
]

trump75_keywords = [
    "Trump", "Donald Trump", "realDonaldTrump", "Trump 2024", "Trump Train",
    "MAGA", "Make America Great Again", "Keep America Great", "Trump Rally",
    "Trump Supporters", "Trump Won", "Stop The Steal", "45th President", "47th President",
    "Trump Campaign", "Trump News", "Trump Speech", "Trump Admin", "Trump Administration",
    "Trump Indictment", "Trump Arrest", "Trump Trial", "Trump Judge",
    "Trump Family", "Ivanka Trump", "Melania Trump", "Eric Trump", "Don Jr",
    "Trump Organization", "Trump Hotel", "Trump Tower", "Trump Voters",
    "Trump Base", "Trumpism", "ProTrump", "AntiTrump", "mar-a-lago",
    "Truth Social", "January 6", "J6", "Capitol riot", "Capitol insurrection",
    "Election fraud", "Election interference", "Georgia case", "Stormy Daniels",
    "Hush money", "Classified documents", "Fake electors", "Trump Lawyers",
    "Giuliani", "Sidney Powell", "Mike Lindell", "Dominion lawsuit", "Fox News Trump",
    "Trump Tweets", "Trump Twitter", "Trump Mugshot", "Trump Booking",
    "Trump Fundraiser", "Trump Rally Crowd", "Trump Voter Fraud", "Big Lie",
    "Impeachment", "Trump Impeachment", "Russia Investigation", "Mueller Report",
    "Trump Tax Returns", "Trump Pardons", "Trump Allies", "Trump Critics",
    "Red Hat", "Trump Base", "Trump Support"
]


politicalnottrump_keywords = [
    "activist", "amendment", "anti-corruption", "assembly", "autocracy", "authoritarian", 
    "Ballot", "biden", "campaign", "candidate", "capitalist", "caucus", "censorship", 
    "city council", "climate policy", "Clinton", "civic", "civil liberties", "council", 
    "Congress", "Congressman", "congresswoman", "conservative", "constitution", "corruption", 
    "Covid", "debate", "Democrat", "democracy", "diplomacy", "election", "equality", 
    "European Union", "executive", "federal", "foreign", "freedom", "governor", "Hamas", 
    "Harris", "healthcare", "Hezbollah", "House of Representatives",
    "infrastructure", "Iran", "Israel", "judiciary", "justice", "Kim Jong Un", "law", "left-wing", 
    "legislation", "liberal", "lobbyist", "mandate", "mayor", "Mcconnell", "military", "Movement", 
    "MP", "nation", "national security", "nationalism", "NATO", "Netanyahu", "Obama", "PAC", 
    "parliament", "party", "patriot", "patriotism", "Pelosi", "polling", "policy", "poverty", 
    "power", "president", "progressive", "public office", "Putin", "rally", "reform", "regulation", 
    "relations", "representative", "Republican", "rights", "sanctions", "Schumer", "Senate", "senator", 
    "socialism", "socialist", "sovereignty", "state", "stimulus", "Supreme court", "Surveillance", 
    "starmer", "taxes", "transparency", "Ukraine", "veto", "vote", 
    "Walz", "welfare", "White house", "Xi JinPing", "Zelensky"
]

politicalnottrumpharris_keywords = [
    "activist", "amendment", "anti-corruption", "assembly", "autocracy", "authoritarian", 
    "Ballot", "biden", "campaign", "candidate", "capitalist", "caucus", "censorship", 
    "city council", "climate policy", "Clinton", "civic", "civil liberties", "council", 
    "Congress", "Congressman", "congresswoman", "conservative", "constitution", "corruption", 
    "Covid", "debate", "Democrat", "democracy", "diplomacy", "election", "equality", 
    "European Union", "executive", "federal", "foreign", "freedom", "governor", "Hamas", "healthcare", "Hezbollah", "House of Representatives", "impeachment", "immigration", 
    "infrastructure", "Iran", "Israel", "judiciary", "justice", "Kim Jong Un", "law", "left-wing", 
    "legislation", "liberal", "lobbyist", "mandate", "mayor", "Mcconnell", "military", "Movement", 
    "MP", "nation", "national security", "nationalism", "NATO", "Netanyahu", "Obama", "PAC", 
    "parliament", "party", "patriot", "patriotism", "Pelosi", "polling", "policy", "poverty", 
    "power", "president", "progressive", "public office", "Putin", "rally", "reform", "regulation", 
    "relations", "representative", "Republican", "rights", "sanctions", "Schumer", "Senate", "senator", 
    "socialism", "socialist", "sovereignty", "state", "stimulus", "Supreme court", "Surveillance", 
    "starmer", "taxes", "transparency", "Ukraine", "veto", "vote", "welfare", "White house", "Xi JinPing", "Zelensky"
]

politicalnotisrael_keywords = [
    "activist", "amendment", "anti-corruption", "assembly", "autocracy", "authoritarian", 
    "Ballot", "biden", "campaign", "candidate", "capitalist", "caucus", "censorship", 
    "city council", "climate policy", "Clinton", "civic", "civil liberties", "council", 
    "Congress", "Congressman", "congresswoman", "conservative", "constitution", "corruption", 
    "Covid", "debate", "Democrat", "democracy", "diplomacy", "election", "equality", 
    "European Union", "executive", "federal", "foreign", "freedom", "governor", 
    "Harris", "healthcare", "House of Representatives", "impeachment", "immigration", 
    "infrastructure", "judiciary", "justice", "Kim Jong Un", "law", "left-wing", 
    "legislation", "liberal", "lobbyist", "mandate", "mayor", "Mcconnell", "military", "Movement", 
    "MP", "nation", "national security", "nationalism", "NATO", "Obama", "PAC", 
    "parliament", "party", "patriot", "patriotism", "Pelosi", "polling", "policy", "poverty", 
    "power", "president", "progressive", "public office", "Putin", "rally", "reform", "regulation", 
    "relations", "representative", "Republican", "rights", "sanctions", "Schumer", "Senate", "senator", 
    "socialism", "socialist", "sovereignty", "state", "stimulus", "Supreme court", "Surveillance", 
    "starmer", "tariffs", "taxes", "transparency", "Trump", "Ukraine", "Vance", "veto", "vote", 
    "Walz", "welfare", "White house", "Xi JinPing", "Zelensky"
]

covid_keywords = ["covid", "coronavirus", "covid19", "SARS-CoV-2",
    "vaccine", "vaccines", "covid vaccine", "covid vaccines", "pandemic", "Wuhan", "china virus", "chinese virus"
                 ]

covid75_keywords = [
    "covid", "covid19", "coronavirus", "SARS-CoV-2", "pandemic", "covid vaccine", "covid vaccines",
    "vaccine", "vaccines", "Wuhan", "china virus", "chinese virus", "quarantine", "lockdown",
    "social distancing", "mask", "masks", "face covering", "PPE", "ventilator", "ICU",
    "flatten the curve", "herd immunity", "asymptomatic", "symptomatic", "long covid",
    "breakthrough infection", "delta variant", "omicron", "variant", "mutations", "transmission",
    "positivity rate", "infection rate", "case count", "contact tracing", "isolation", "antibodies",
    "booster", "booster shot", "mrna", "pfizer", "moderna", "johnson & johnson", "astrazeneca",
    "cdc", "who", "fda", "public health", "epidemic", "outbreak", "testing", "rapid test",
    "PCR test", "temperature check", "hand sanitizer", "immunization", "vaccine mandate",
    "vaccine passport", "vaccine hesitancy", "anti-vax", "plandemic", "lab leak", "gain of function",
    "bioweapon", "covid hoax", "covid misinformation", "covid disinformation", "fake news",
    "health emergency", "essential workers", "frontline workers", "remote work", "school closures",
    "stimulus checks", "telehealth"
]


healthsciencecovid_keywords = [
    "ADHD", "AIDS", "AI", "CDC", "CRISPR", "FDA", "GMOs", "HIV", "ICU", "NASA", "NIH", 
    "NSF", "PTSD", "WHO", "acute", "adverse events", "allergy", "analysis", "anxiety", 
    "antibiotic", "antiviral", "archaeology", "artificial intelligence", "asthma", 
    "astronomy", "autoimmune", "autism", "bias", "big bang", "biology", "booster", 
    "cancer", "cause", "causation", "cdc", "chemistry", "china virus", "chinese virus", 
    "chronic", "clinic", "clinical", "clinical trial", "cold", "computer science", 
    "condition", "consensus", "correlation", "coronavirus", "covid", "covid vaccine", 
    "covid vaccines", "covid19", "data", "death", "depression", "diagnosis", "disease", 
    "doctor", "double-blind", "drug", "ecology", "efficacy", "emergency room", 
    "engineering", "epidemic", "evidence", "evolution", "experiment", "experiments", 
    "fake cure", "fieldwork", "findings", "flu", "falsifiable", "gene editing", "genetics", 
    "geology", "health", "healthcare", "heart disease", "hospital", "hypothesis", 
    "illness", "influenza", "institute", "insulin", "jab", "journal", "lab", "laboratory", 
    "long covid", "machine learning", "margin of error", "mathematics", "medical", 
    "medication", "medicine", "mental health", "meteorology", "method", "methodology", 
    "microchip", "model", "mRNA", "ncbi", "neuroscience", "nih", "nlm", "nurse", 
    "obesity", "observational", "open access", "outbreak", "pandemic", "patient", 
    "peer review", "peer-reviewed", "physics", "placebo", "pmc", "preprint", 
    "psychology", "public health", "publication", "pseudoscience", "quantum", 
    "replication", "research", "results", "robotics", "safety compliance", "sample size", 
    "science", "scientific", "scientist", "side effects", "significant", "simulation", 
    "sociology", "statistically significant", "statistics", "stroke", "study", "studies", 
    "symptom", "theory", "theory of relativity", "treatment", "university", "vaccine", 
    "vaccinated", "vaccination", "vaccines", "variance", "wuhan", "wellness"
]

healthsciencenotpolitical_keywords = [
    "ADHD", "AIDS", "AI", "CRISPR", "FDA", "GMOs", "HIV", "ICU", "NASA", 
    "NSF", "PTSD", "acute", "adverse events", "allergy", "analysis", "anxiety", 
    "antibiotic", "antiviral", "archaeology", "artificial intelligence", "asthma", 
    "astronomy", "autoimmune", "autism", "big bang", "biology", "booster", 
    "cancer", "cause", "causation", "chemistry",
    "chronic", "clinic", "clinical", "clinical trial", "cold", "computer science", 
    "condition", "consensus", "correlation", 
    "data", "death", "depression", "diagnosis", "disease", 
    "doctor", "double-blind", "drug", "ecology", "efficacy", "emergency room", 
    "engineering", "epidemic", "evidence", "evolution", "experiment", "experiments", 
    "fieldwork", "findings", "flu", "falsifiable", "gene editing", "genetics", 
    "geology", "health", "healthcare", "heart disease", "hospital", "hypothesis", 
    "illness", "influenza", "institute", "insulin", "jab", "journal", "lab", "laboratory", 
   "machine learning", "margin of error", "mathematics", "medical", 
    "medication", "medicine", "mental health", "meteorology", "method", "methodology", 
    "microchip", "model", "ncbi", "neuroscience", "nlm", "nurse", 
    "obesity", "observational", "open access", "outbreak", "patient", 
    "peer review", "peer-reviewed", "physics", "placebo", "pmc", "preprint", 
    "psychology", "public health", "publication", "pseudoscience", "quantum", 
    "replication", "research", "results", "robotics", "safety compliance", "sample size", 
    "science", "scientific", "scientist", "side effects", "significant", "simulation", 
    "sociology", "statistically significant", "statistics", "stroke", "study", "studies", 
    "symptom", "theory", "theory of relativity", "treatment", "university", 
    "variance","wellness"
]

sex_keywords = [
    "sex", "sexual", "sexy", "nude", "nudes", "nsfw", "porn", "porno", "pornographic", "xxx",
    "fetish", "fetishes", "kink", "kinky", "bdsm", "dominatrix", "submissive", "dominant", "roleplay",
    "erotic", "erotica", "orgasm", "climax", "masturbate", "masturbation", "solo", "handjob", "blowjob",
    "oral", "anal", "penetration", "vaginal", "cum", "cumming", "ejaculate", "ejaculation", "moan", "moaning",
    "wet", "horny", "naughty", "nipple", "nipples", "boobs", "tits", "breasts", "ass", "butt", "butthole",
    "genitals", "penis", "dick", "cock", "balls", "scrotum", "testicles", "vagina", "pussy", "clit", "clitoris",
    "fuck", "fucking", "screw", "bang", "hookup", "hook up", "one night stand", "make out", "lick", "licking",
    "dirty talk", "sext", "sexting", "sex tape", "onlyfans"
]

sextargeted_keywords = [
    "sex", "sexual", "sexting", "sext", "porn", "pornographic", "nude", "nudes",
    "onlyfans", "sex tape", "masturbation", "orgasm", "clitoris", "vagina", "penis",
    "genitals", "erection", "ejaculation", "hookup", "hook up", "one night stand",
    "sex education", "sex ed", "sexual health", "sexual misinformation",
    "sexual abuse", "sex trafficking", "child grooming", "explicit content"
]


earthquake_keywords = [
    "earthquake", "quake", "tremor", "aftershock", "foreshock", "seismic", "seismology", "seismograph", 
    "seismometer", "epicenter", "epicentre", "hypocenter", "fault line", "fault", "tectonic", 
    "tectonic plates", "plate tectonics", "ground shaking", "ground motion", "vibration", 
    "magnitude", "richter scale", "moment magnitude", "intensity", "mercalli scale", 
    "rupture", "fault rupture", "ground rupture", "subduction", "subduction zone", "strike-slip", 
    "normal fault", "reverse fault", "lateral fault", "shear zone", "elastic rebound", 
    "crustal movement", "ground displacement", "tsunami", "aftershocks", "pre-shock", "shockwave", 
    "P wave", "S wave", "surface wave", "body wave", "earth movement", "seismic activity", 
    "seismic zone", "earthquake swarm", "liquefaction", "earthquake preparedness", 
    "emergency response", "disaster relief", "geological hazard", "natural disaster", 
    "earthquake drill", "earthquake damage", "collapsed building", "structural failure", 
    "evacuation", "trembling", "geophysics", "displacement", "tectonic shift", 
    "earthquake warning", "early warning system", "shake alert", "aftershock sequence", 
    "tectonic boundary", "continental drift", "volcanic earthquake", "earthquake forecast", 
    "quake epicenter", "earthquake prediction", "earthquake predictions"
]

earthquakesmall_keywords = [
    "earthquake", "quake", "aftershock", "foreshock", "tremor",
    "seismic", "seismograph", "seismometer", "epicenter", "epicentre",
    "fault line", "tectonic plates", "plate tectonics", "richter scale",
    "moment magnitude", "mercalli scale", "subduction zone",
    "earthquake swarm", "earthquake damage", "earthquake prediction",
    "earthquake predictions", "earthquake warning", "shake alert",
    "liquefaction", "tsunami", "rupture", "ground rupture"
]


nutrition_keywords = [
    "nutrition", "nutrients", "macronutrients", "micronutrients",
    "protein", "carbohydrates", "carbs", "fat", "fats", "trans fats",
    "saturated fat", "unsaturated fat", "cholesterol", "fiber", "sugar",
    "added sugar", "sodium", "salt", "calories", "vitamins", "vitamin A",
    "vitamin B12", "vitamin C", "vitamin D", "vitamin E", "vitamin K",
    "minerals", "iron", "calcium", "magnesium", "zinc", "iodine", "folate",
    "antioxidants", "probiotics", "prebiotics", "omega-3", "omega-6",
    "glycemic index", "insulin", "blood sugar", "insulin resistance",
    "malnutrition", "overnutrition", "undernutrition", "deficiency", 
    "nutrient deficiency", "nutrition science", "nutrition facts",
    "nutrition label", "daily value", "recommended intake", 
    "nutritionist", "dietitian", "registered dietitian", "supplements",
    "multivitamin", "nutrition misinformation", "nutrition myth"
]

nutritionexercise_keywords = [
    # Diet and nutrition
    "nutrition", "nutrients", "macronutrients", "micronutrients",
    "protein", "carbohydrates", "carbs", 
    "trans fats", "saturated fat", "unsaturated fat",
    "cholesterol", "added sugar", 
    "vitamins", "vitamin A", "vitamin B12", "vitamin C", "vitamin D",
    "vitamin E", "vitamin K", "minerals", "iron", "calcium", 
    "magnesium", "zinc", "iodine", "folate",
    "antioxidants", "probiotics", "prebiotics", "omega-3", "omega-6",
    "glycemic index", "insulin resistance", "blood sugar", "insulin",
    "malnutrition", "overnutrition", "undernutrition", 
    "nutrient deficiency", "nutrition science", "nutrition facts",
    "nutrition label", "daily value", "recommended intake", 
    "nutritionist", "dietitian", "registered dietitian", 
    "supplements", "multivitamin", 
    "nutrition misinformation", "nutrition myth",
    
    # Exercise & fitness
    "exercise", "physical activity", "fitness", "strength training",
    "resistance training", "weightlifting", "cardio", "aerobic", "HIIT",
    "interval training", "workout", "fitness routine", "fitness myth",

    # Yoga & lifestyle
    "yoga", "yoga practice", "mind-body", "wellness", "lifestyle change",
    "healthy lifestyle", "health transformation", "fitness transformation",
    "diet change", "clean eating", "health journey"
]


cancer_keywords = [
    "cancer", "tumor", "malignancy", "carcinoma", "sarcoma", "lymphoma", "leukemia", "melanoma",
    "oncology", "metastasis", "neoplasm", "chemotherapy", "radiation therapy", "immunotherapy",
    "targeted therapy", "biopsy", "stage 4 cancer", "benign tumor", "malignant tumor",
    "breast cancer", "lung cancer", "prostate cancer", "colorectal cancer", "pancreatic cancer",
    "ovarian cancer", "skin cancer", "brain tumor", "cervical cancer", "testicular cancer",
    "thyroid cancer", "kidney cancer", "bone cancer", "bladder cancer", "liver cancer",
    "cancer screening", "genetic mutation",
    "BRCA", "oncogene", "tumor suppressor gene", "palliative care", "cancer survivor", "remission","cancer risk", "carcinogen", "environmental exposure",
    "radiotherapy", "cancer vaccine", "oncologist"
]

animal_keywords = [
    # Specific attack phrases
    "shark attack", "bear attack", "mountain lion", "cougar attack",
    "alligator attack", "crocodile attack", "snake bite", "python attack",
    "wolf attack", "coyote attack", "dog attack", "wild animal attack",
    "lion attack", "tiger attack", "elephant rampage", "hippo attack",
    "animal mauling", "zoo escape", "escaped animal", "predator attack",
    "killer animal", "man-eating", "deadly bite", "fatal animal attack",
    "wildlife attack", "attacked by animal", "animal kills", "animal horror",

    # Broader animal terms (without "attack")
    "shark", "bear", "mountain lion", "cougar", "alligator", "crocodile",
    "snake", "python", "wolf", "coyote", "lion", "tiger", "elephant",
    "hippo", "gator", "wild animal", "predator", "zoo", "escaped animal",
    "mauling", "animal bite"
]

In [38]:
# call function for each keyword list. 
# Takes about 30 minutes to run:

keywords = ['political', 'israel', 'trump', 'covid', 'earthquakesmall', 'cancer']

for keyword in keywords:
    add_keyword_column(df, globals()[f'{keyword}_keywords'], keyword)

df.to_csv('df.csv', index=False)